#### The code to focus on the best performing models (Extra Trees, Random Forest, and Decision Tree) and implement the three different data splitting scenarios

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
import time
from pathlib import Path
import os
from datetime import datetime

# For basemap
try:
    import contextily as ctx
    import geopandas as gpd
    from shapely.geometry import Point
    BASEMAP_AVAILABLE = True
except ImportError:
    BASEMAP_AVAILABLE = False
    print("Warning: contextily and/or geopandas not available. Maps will be generated without basemap.")

warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

class GroundwaterPredictor:
    """Professional groundwater depth prediction analysis with temporal well splitting and detailed visualizations."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }
        
        self.train_indices = []
        self.test_indices = []
        
        # --- PERFORMANCE THRESHOLDS ---
        self.ape_threshold = 30.0 # No longer used for classification, but kept for potential reference
        self.absolute_error_threshold = 5.0 # The key threshold for poor performance
        
        self.start_time = None
        self.timing_log = []
        
        run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.output_dir = Path(f"groundwater_analysis_results_{run_timestamp}")
        self.output_dir.mkdir(exist_ok=True)
        print(f"📂 Output will be saved to: {self.output_dir}")
        
    def log_time(self, step_name):
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
        
    def print_timing_summary(self):
        if not self.timing_log: return
        print("\n" + "="*60 + "\n⏰ TIMING SUMMARY\n" + "="*60)
        for i, entry in enumerate(self.timing_log):
            step_time = entry['elapsed_minutes'] - (self.timing_log[i-1]['elapsed_minutes'] if i > 0 else 0)
            print(f"{entry['step']:<40} | {step_time:>8.2f} min | {entry['timestamp']}")
        total_time = self.timing_log[-1]['elapsed_minutes']
        print("-" * 60 + f"\n{'TOTAL RUNTIME':<40} | {total_time:>8.2f} min\n" + "="*60)
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        non_zero_mask = y_true != 0
        if not np.any(non_zero_mask): return 0.0
        return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
    
    def absolute_percentage_error(self, y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        return np.where(y_true != 0, np.abs((y_true - y_pred) / y_true) * 100, np.inf)

    # <<< MODIFIED CODE START >>>
    def _classify_performance(self, y_true, y_pred):
        """
        Classify predictions based on absolute error.
        A prediction is 'Poor' if the absolute error is > 5m.
        """
        abs_error = np.abs(np.array(y_true) - np.array(y_pred))
        
        # --- UPDATED LOGIC: Only consider absolute error for performance classification ---
        poor_mask = abs_error > self.absolute_error_threshold
        good_mask = ~poor_mask
        return good_mask, poor_mask
    # <<< MODIFIED CODE END >>>
        
    def load_and_preprocess_data(self):
        print("🌊 Chile Groundwater Depth Prediction Analysis\n" + "=" * 60)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")
        self.log_time("Analysis Started")
        
        try:
            if str(self.file_path).lower().endswith('.csv'): self.df = pd.read_csv(self.file_path)
            elif os.path.exists(self.file_path): self.df = pd.read_excel(self.file_path)
            else: raise FileNotFoundError(f"File not found at {self.file_path}")
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}"); return False
        self.log_time("Data Loading")
        
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered to {len(self.df_filtered)} rows where Status is blank")
        
        initial_rows = len(self.df_filtered)
        self.df_filtered.dropna(subset=[self.target_col], inplace=True)
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")

        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered[self.df_filtered[self.target_col] > 0].copy()
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with non-positive target values (Depth to water <= 0)")
        print(f"Final dataset shape for analysis: {self.df_filtered.shape}")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        return True
        
    def _prepare_features(self):
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        if self.target_col in numeric_cols: numeric_cols.remove(self.target_col)
        
        cols_to_exclude = ['Status', 'Code', 'Date_String']
        numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]
        categorical_cols = [c for c in categorical_cols if c not in cols_to_exclude]
        
        for col in numeric_cols: self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        encoded_cols = []
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
                le = LabelEncoder()
                encoded_col_name = col + '_encoded'
                self.df_filtered[encoded_col_name] = le.fit_transform(self.df_filtered[col].astype(str))
                encoded_cols.append(encoded_col_name)
        
        self.feature_cols = numeric_cols + encoded_cols
        print(f"Final feature set: {len(self.feature_cols)} features")

    def create_temporal_well_split(self, X, y, train_ratio=0.7):
        if 'Code' not in self.df_filtered.columns or 'Date_String' not in self.df_filtered.columns:
            print("Warning: 'Code' or 'Date_String' not found. Using random split.")
            return train_test_split(X, y, test_size=1 - train_ratio, random_state=42)

        df_temp = self.df_filtered.copy()
        df_temp['Date_String'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        df_temp.dropna(subset=['Date_String'], inplace=True)
        
        train_indices, test_indices = [], []
        for _, group in df_temp.groupby('Code'):
            group_sorted = group.sort_values('Date_String')
            indices = group_sorted.index.tolist()
            if len(indices) <= 9: train_indices.extend(indices)
            else:
                n_train = max(1, int(len(indices) * train_ratio))
                train_indices.extend(indices[:n_train])
                test_indices.extend(indices[n_train:])
        
        self.train_indices = self.df_filtered.index.intersection(train_indices)
        self.test_indices = self.df_filtered.index.intersection(test_indices)
        
        train_mask = self.df_filtered.index.isin(self.train_indices)
        test_mask = self.df_filtered.index.isin(self.test_indices)
        
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]
        
    def run_analysis(self):
        if self.df_filtered is None or len(self.df_filtered) == 0:
            print("❌ Data not loaded or empty. Aborting analysis.")
            return

        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        print(f"\n{'='*60}\n🔄 Analysis with Temporal Well Split\n{'='*60}")
        # <<< MODIFIED TEXT >>>
        print(f"📊 Performance Criterion: A prediction is 'Poor' if |Error| > {self.absolute_error_threshold}m.")
        
        try:
            X_train, X_test, y_train, y_test = self.create_temporal_well_split(X, y)
            if len(X_train) == 0 or len(X_test) == 0:
                print("⚠️ Warning: Temporal split resulted in an empty train or test set. Aborting."); return
            print(f"Training samples: {len(X_train)} ({len(X_train)/len(y)*100:.1f}%) | Testing samples: {len(X_test)} ({len(X_test)/len(y)*100:.1f}%)")
            
            self._generate_data_split_distribution_plot(y_train, y_test)
            self._generate_temporal_distribution_plots()
            results = self._train_and_evaluate_models(X_train, X_test, y_train, y_test)
            self._generate_one_to_one_plots(results, y_train, y_test)
            self._generate_testing_performance_plots(results, y_test)
            self._generate_performance_distribution_plots(results, y_test)
            self._generate_well_location_maps_and_save_results(results, y_test)
            
        except Exception as e:
            print(f"❌ ERROR during Analysis Pipeline: {e}"); import traceback; traceback.print_exc()
            
        self.log_time("Full Analysis Pipeline")
        self.print_timing_summary()
        print(f"\n🎉 Analysis completed! Results saved in '{self.output_dir}' directory")

    def _train_and_evaluate_models(self, X_train, X_test, y_train, y_test):
        results = {}
        for name, model in self.models.items():
            print(f"  Training {name}..."); model_start_time = time.time()
            model.fit(X_train, y_train)
            model_time = (time.time() - model_start_time) / 60
            print(f"    ✅ {name} completed in {model_time:.2f} minutes")
            results[name] = {
                'model': model,
                'train_metrics': self._calculate_metrics(y_train, model.predict(X_train)),
                'test_metrics': self._calculate_metrics(y_test, model.predict(X_test)),
                'train_predictions': model.predict(X_train),
                'test_predictions': model.predict(X_test),
            }
        return results
    
    def _calculate_metrics(self, y_true, y_pred):
        return {
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }

    def _get_stats_text(self, data, title=""):
        if len(data) == 0: return f"{title}\nN = 0"
        stats = pd.Series(data).describe()
        return (f"{title}\nN = {int(stats['count'])}\nMean = {stats['mean']:.2f}\n"
                f"Std = {stats['std']:.2f}\nMin = {stats['min']:.2f}\nMax = {stats['max']:.2f}")

    def _generate_data_split_distribution_plot(self, y_train, y_test):
        print("  Generating data split distribution plot...")
        plt.figure(figsize=(10, 6))
        sns.histplot(y_train, color="dodgerblue", label='Training Set', kde=True, stat="density", linewidth=0)
        sns.histplot(y_test, color="darkorange", label='Testing Set', kde=True, stat="density", linewidth=0)
        plt.title('Distribution of Depth to Water in Training and Testing Sets'); plt.xlabel('Depth to Water (m)'); plt.ylabel('Density')
        props = dict(boxstyle='round', facecolor='white', alpha=0.8)
        plt.text(0.95, 0.95, self._get_stats_text(y_train, 'Training Set'), transform=plt.gca().transAxes, fontsize=10, va='top', ha='right', bbox=props)
        plt.text(0.95, 0.70, self._get_stats_text(y_test, 'Testing Set'), transform=plt.gca().transAxes, fontsize=10, va='top', ha='right', bbox=props)
        plt.legend(); plt.tight_layout(); plt.savefig(self.output_dir / 'data_split_distribution.png', facecolor='white'); plt.close()

    def _generate_temporal_distribution_plots(self):
        print("  Generating temporal distribution plots...")
        if 'Date_String' not in self.df_filtered.columns: print("  Skipping temporal plots: 'Date_String' column not found."); return

        df_temp = self.df_filtered[['Date_String', self.target_col]].copy()
        df_temp['Date'] = pd.to_datetime(df_temp['Date_String'], errors='coerce'); df_temp.dropna(subset=['Date'], inplace=True)
        df_temp['Year'] = df_temp['Date'].dt.year
        
        train_df = df_temp.loc[self.train_indices]; test_df = df_temp.loc[self.test_indices]
        fig, axes = plt.subplots(2, 1, figsize=(15, 12))
        counts_df = pd.DataFrame({'Training': train_df['Year'].value_counts(), 'Testing': test_df['Year'].value_counts()}).sort_index().fillna(0)
        counts_df.plot(kind='bar', ax=axes[0], stacked=True, color=['skyblue', 'salmon'])
        axes[0].set_title('Number of Samples per Year'); axes[0].set_ylabel('Number of Samples'); axes[0].tick_params(axis='x', rotation=45)
        
        combined_df = pd.concat([train_df.assign(Set='Training'), test_df.assign(Set='Testing')])
        sns.boxplot(x='Year', y=self.target_col, hue='Set', data=combined_df, ax=axes[1], palette={'Training': 'skyblue', 'Testing': 'salmon'})
        axes[1].set_title('Distribution of Depth to Water by Year'); axes[1].set_ylabel('Depth to Water (m)'); axes[1].tick_params(axis='x', rotation=45)
        plt.tight_layout(); plt.savefig(self.output_dir / 'temporal_distribution.png', facecolor='white'); plt.close()

    def _generate_one_to_one_plots(self, results, y_train, y_test):
        print("  Generating one-to-one plots...")
        for model_name, res in results.items():
            self._plot_one_to_one(y_train, res['train_predictions'], res['train_metrics'], model_name, 'Training')
            self._plot_one_to_one(y_test, res['test_predictions'], res['test_metrics'], model_name, 'Testing')

    def _plot_one_to_one(self, y_true, y_pred, metrics, model_name, data_type):
        plt.figure(figsize=(8, 8))
        ax = sns.scatterplot(x=y_true, y=y_pred, alpha=0.6, color='steelblue', edgecolor='w', s=50)
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'r--', lw=2, label='1:1 Line')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel('Actual Depth to Water (m)'); ax.set_ylabel('Predicted Depth to Water (m)')
        ax.set_title(f'{model_name}: {data_type} Set\nPredicted vs Actual')
        text_str = (f"R² = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.2f} m\n"
                    f"MAPE = {metrics['mape']:.2f}%\nN = {len(y_true)}")
        props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='black')
        ax.text(0.05, 0.95, text_str, transform=ax.transAxes, fontsize=11, va='top', bbox=props)
        plt.legend(); plt.tight_layout()
        filename = f'{model_name.lower().replace(" ", "_")}_{data_type.lower()}_one_to_one.png'
        plt.savefig(self.output_dir / filename, bbox_inches='tight', facecolor='white'); plt.close()
        
    def _generate_testing_performance_plots(self, results, y_test):
        print(f"  Generating testing sample performance plots...")
        fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True, sharey=True)
        # <<< MODIFIED TEXT >>>
        fig.suptitle(f'Testing Performance (Poor if |Error| > {self.absolute_error_threshold}m)', fontsize=16, fontweight='bold')
        
        for i, (model_name, res) in enumerate(results.items()):
            good_mask, poor_mask = self._classify_performance(y_test, res['test_predictions'])
            ax = axes[i]
            ax.scatter(y_test[good_mask], res['test_predictions'][good_mask], alpha=0.6, color='green', s=30, label=f'Good (N={good_mask.sum()})')
            ax.scatter(y_test[poor_mask], res['test_predictions'][poor_mask], alpha=0.8, color='red', s=40, label=f'Poor (N={poor_mask.sum()})')
            lims = [min(y_test.min(), res['test_predictions'].min()), max(y_test.max(), res['test_predictions'].max())]
            ax.plot(lims, lims, 'k--', lw=2, alpha=0.7, label='1:1 Line'); ax.set_xlim(lims); ax.set_ylim(lims)
            ax.set_xlabel('Actual Depth to Water (m)'); ax.set_title(f'{model_name}'); ax.legend()
            if i == 0: ax.set_ylabel('Predicted Depth to Water (m)')
            ax.text(0.02, 0.98, f'{good_mask.sum() / len(y_test) * 100:.1f}% Good', transform=ax.transAxes, fontsize=10, va='top', bbox=dict(boxstyle='round,pad=0.3', fc='lightgreen', alpha=0.7))
        
        plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.savefig(self.output_dir / 'testing_performance_scatter.png', facecolor='white'); plt.close()

    def _generate_performance_distribution_plots(self, results, y_test):
        print("  Generating performance-based distribution plots...")
        fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True, sharey=True)
        fig.suptitle('Distribution of Actual Depths for Good vs. Poor Predictions', fontsize=16, fontweight='bold')
        for i, (model_name, res) in enumerate(results.items()):
            good_mask, poor_mask = self._classify_performance(y_test, res['test_predictions'])
            ax = axes[i]
            sns.histplot(y_test[good_mask], ax=ax, color='green', label='Good', kde=True, stat="density", lw=0, alpha=0.6)
            sns.histplot(y_test[poor_mask], ax=ax, color='red', label='Poor', kde=True, stat="density", lw=0, alpha=0.6)
            ax.set_title(model_name); ax.set_xlabel('Actual Depth to Water (m)'); ax.legend()
            if i == 0: ax.set_ylabel('Density')
            props = dict(boxstyle='round', facecolor='white', alpha=0.8)
            ax.text(0.98, 0.98, self._get_stats_text(y_test[good_mask], 'Good'), transform=ax.transAxes, fontsize=9, va='top', ha='right', bbox=props)
            ax.text(0.98, 0.65, self._get_stats_text(y_test[poor_mask], 'Poor'), transform=ax.transAxes, fontsize=9, va='top', ha='right', bbox=props)
        plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.savefig(self.output_dir / 'performance_distribution_by_model.png', facecolor='white'); plt.close()

    def _generate_well_location_maps_and_save_results(self, results, y_test):
        print("  Generating well location maps and saving results to Excel...")
        coord_cols = ['Longitude_GCS_WGS_1984', 'Latitude_GCS_WGS_1984']
        if not all(col in self.df_filtered.columns for col in coord_cols):
            print("  Warning: Coordinate columns not found. Skipping map generation."); return
        
        test_data_with_coords = self.df_filtered.loc[self.test_indices].copy()
        if len(test_data_with_coords) != len(y_test):
            print(f"  Warning: Mismatch in data lengths. Skipping maps."); return
        
        excel_data = []
        fig, axes = plt.subplots(1, 3, figsize=(24, 8))
        # <<< MODIFIED TEXT >>>
        fig.suptitle(f'Well Performance Map (Poor if |Error| > {self.absolute_error_threshold}m)', fontsize=16, fontweight='bold')
        
        for i, (model_name, res) in enumerate(results.items()):
            test_data_copy = test_data_with_coords.copy()
            test_data_copy['predicted'] = res['test_predictions']
            _, poor_mask = self._classify_performance(test_data_copy[self.target_col], test_data_copy['predicted'])
            test_data_copy['Performance_Category'] = np.where(poor_mask, 'Poor', 'Good')
            excel_data.extend(test_data_copy.assign(Model=model_name).to_dict('records'))

            well_performance = test_data_copy.groupby('Code').agg({
                coord_cols[0]: 'mean', coord_cols[1]: 'mean',
                'Performance_Category': lambda x: 'Poor' if 'Poor' in x.values else 'Good'
            }).reset_index()

            ax = axes[i]
            good_wells = well_performance[well_performance['Performance_Category'] == 'Good']
            poor_wells = well_performance[well_performance['Performance_Category'] == 'Poor']

            if BASEMAP_AVAILABLE and not well_performance.empty:
                try:
                    gdf = gpd.GeoDataFrame(well_performance, geometry=gpd.points_from_xy(well_performance[coord_cols[0]], well_performance[coord_cols[1]]), crs="EPSG:4326").to_crs(epsg=3857)
                    gdf[gdf['Performance_Category'] == 'Good'].plot(ax=ax, color='green', alpha=0.7, markersize=60, label=f'Good Wells (N={len(good_wells)})')
                    gdf[gdf['Performance_Category'] == 'Poor'].plot(ax=ax, color='red', alpha=0.8, markersize=80, label=f'Poor Wells (N={len(poor_wells)})')
                    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.8)
                except Exception as e: print(f"  Warning: Basemap error for {model_name}: {e}")
            else:
                ax.scatter(good_wells[coord_cols[0]], good_wells[coord_cols[1]], c='green', label=f'Good Wells (N={len(good_wells)})')
                ax.scatter(poor_wells[coord_cols[0]], poor_wells[coord_cols[1]], c='red', label=f'Poor Wells (N={len(poor_wells)})')
            
            ax.set_title(f'{model_name}'); ax.set_xlabel('Longitude'); ax.legend()
            if i == 0: ax.set_ylabel('Latitude')
        
        plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.savefig(self.output_dir / 'well_location_performance_maps.png', facecolor='white'); plt.close()
        
        if excel_data:
            df = pd.DataFrame(excel_data)
            df['ape'] = self.absolute_percentage_error(df[self.target_col], df['predicted'])
            df['abs_error'] = np.abs(df[self.target_col] - df['predicted'])
            excel_path = self.output_dir / "detailed_testing_results.xlsx"
            try:
                df.to_excel(excel_path, sheet_name='All_Results', index=False, engine='openpyxl')
                print(f"  ✅ Detailed results saved to {excel_path}")
            except Exception as e: print(f"  ❌ Could not save detailed Excel file: {e}")

# ==============================================================================
# MAIN EXECUTION BLOCK
# ==============================================================================
def main():
    # IMPORTANT: Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    if not os.path.exists(file_path):
        print(f"❌ FATAL ERROR: Input file not found at '{file_path}'")
        print("Please update the 'file_path' variable in the main() function.")
        return

    predictor = GroundwaterPredictor(file_path)
    if predictor.load_and_preprocess_data():
        predictor.run_analysis()

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
import time
from pathlib import Path
import os
from datetime import datetime

# For basemap
try:
    import contextily as ctx
    import geopandas as gpd
    from shapely.geometry import Point
    BASEMAP_AVAILABLE = True
except ImportError:
    BASEMAP_AVAILABLE = False
    print("Warning: contextily and/or geopandas not available. Maps will be generated without basemap.")

warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

class GroundwaterPredictor:
    """Professional groundwater depth prediction analysis with temporal well splitting and detailed visualizations."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }
        
        self.train_indices = []
        self.test_indices = []
        
        # --- PERFORMANCE THRESHOLDS ---
        self.ape_threshold = 30.0 # No longer used for classification, but kept for potential reference
        self.absolute_error_threshold = 5.0 # The key threshold for poor performance
        
        self.start_time = None
        self.timing_log = []
        
        run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.output_dir = Path(f"groundwater_analysis_results_{run_timestamp}")
        self.output_dir.mkdir(exist_ok=True)
        print(f"📂 Output will be saved to: {self.output_dir}")
        
    def log_time(self, step_name):
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
        
    def print_timing_summary(self):
        if not self.timing_log: return
        print("\n" + "="*60 + "\n⏰ TIMING SUMMARY\n" + "="*60)
        for i, entry in enumerate(self.timing_log):
            step_time = entry['elapsed_minutes'] - (self.timing_log[i-1]['elapsed_minutes'] if i > 0 else 0)
            print(f"{entry['step']:<40} | {step_time:>8.2f} min | {entry['timestamp']}")
        total_time = self.timing_log[-1]['elapsed_minutes']
        print("-" * 60 + f"\n{'TOTAL RUNTIME':<40} | {total_time:>8.2f} min\n" + "="*60)
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        non_zero_mask = y_true != 0
        if not np.any(non_zero_mask): return 0.0
        return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
    
    def absolute_percentage_error(self, y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        return np.where(y_true != 0, np.abs((y_true - y_pred) / y_true) * 100, np.inf)

    def _classify_performance(self, y_true, y_pred):
        """
        Classify predictions based on absolute error.
        A prediction is 'Poor' if the absolute error is > 5m.
        """
        abs_error = np.abs(np.array(y_true) - np.array(y_pred))
        
        # --- UPDATED LOGIC: Only consider absolute error for performance classification ---
        poor_mask = abs_error > self.absolute_error_threshold
        good_mask = ~poor_mask
        return good_mask, poor_mask
        
    def load_and_preprocess_data(self):
        print("🌊 Chile Groundwater Depth Prediction Analysis\n" + "=" * 60)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")
        self.log_time("Analysis Started")
        
        try:
            if str(self.file_path).lower().endswith('.csv'): 
                self.df = pd.read_csv(self.file_path)
            elif os.path.exists(self.file_path): 
                self.df = pd.read_excel(self.file_path)
            else: 
                raise FileNotFoundError(f"File not found at {self.file_path}")
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return False
        self.log_time("Data Loading")
        
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered to {len(self.df_filtered)} rows where Status is blank")
        
        initial_rows = len(self.df_filtered)
        self.df_filtered.dropna(subset=[self.target_col], inplace=True)
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")

        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered[self.df_filtered[self.target_col] > 0].copy()
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with non-positive target values (Depth to water <= 0)")
        print(f"Final dataset shape for analysis: {self.df_filtered.shape}")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        return True
        
    def _prepare_features(self):
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        if self.target_col in numeric_cols: numeric_cols.remove(self.target_col)
        
        cols_to_exclude = ['Status', 'Code', 'Date_String']
        numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]
        categorical_cols = [c for c in categorical_cols if c not in cols_to_exclude]
        
        for col in numeric_cols: 
            self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        encoded_cols = []
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
                le = LabelEncoder()
                encoded_col_name = col + '_encoded'
                self.df_filtered[encoded_col_name] = le.fit_transform(self.df_filtered[col].astype(str))
                encoded_cols.append(encoded_col_name)
        
        self.feature_cols = numeric_cols + encoded_cols
        print(f"Final feature set: {len(self.feature_cols)} features")

    def create_temporal_well_split(self, X, y, train_ratio=0.7):
        if 'Code' not in self.df_filtered.columns or 'Date_String' not in self.df_filtered.columns:
            print("Warning: 'Code' or 'Date_String' not found. Using random split.")
            return train_test_split(X, y, test_size=1 - train_ratio, random_state=42)

        df_temp = self.df_filtered.copy()
        df_temp['Date_String'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        df_temp.dropna(subset=['Date_String'], inplace=True)
        
        train_indices, test_indices = [], []
        for _, group in df_temp.groupby('Code'):
            group_sorted = group.sort_values('Date_String')
            indices = group_sorted.index.tolist()
            if len(indices) <= 9: 
                train_indices.extend(indices)
            else:
                n_train = max(1, int(len(indices) * train_ratio))
                train_indices.extend(indices[:n_train])
                test_indices.extend(indices[n_train:])
        
        self.train_indices = self.df_filtered.index.intersection(train_indices)
        self.test_indices = self.df_filtered.index.intersection(test_indices)
        
        train_mask = self.df_filtered.index.isin(self.train_indices)
        test_mask = self.df_filtered.index.isin(self.test_indices)
        
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]
        
    def run_analysis(self):
        if self.df_filtered is None or len(self.df_filtered) == 0:
            print("❌ Data not loaded or empty. Aborting analysis.")
            return

        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        print(f"\n{'='*60}\n🔄 Analysis with Temporal Well Split\n{'='*60}")
        print(f"📊 Performance Criterion: A prediction is 'Poor' if |Error| > {self.absolute_error_threshold}m.")
        
        try:
            X_train, X_test, y_train, y_test = self.create_temporal_well_split(X, y)
            if len(X_train) == 0 or len(X_test) == 0:
                print("⚠️ Warning: Temporal split resulted in an empty train or test set. Aborting.")
                return
            print(f"Training samples: {len(X_train)} ({len(X_train)/len(y)*100:.1f}%) | Testing samples: {len(X_test)} ({len(X_test)/len(y)*100:.1f}%)")
            
            self._generate_data_split_distribution_plot(y_train, y_test)
            self._generate_temporal_distribution_plots()
            results = self._train_and_evaluate_models(X_train, X_test, y_train, y_test)
            self._generate_one_to_one_plots(results, y_train, y_test)
            self._generate_testing_performance_plots(results, y_test)
            self._generate_performance_distribution_plots(results, y_test)
            self._generate_well_location_maps_and_save_results(results, y_test)
            
        except Exception as e:
            print(f"❌ ERROR during Analysis Pipeline: {e}")
            import traceback
            traceback.print_exc()
            
        self.log_time("Full Analysis Pipeline")
        self.print_timing_summary()
        print(f"\n🎉 Analysis completed! Results saved in '{self.output_dir}' directory")

    def _train_and_evaluate_models(self, X_train, X_test, y_train, y_test):
        results = {}
        for name, model in self.models.items():
            print(f"  Training {name}...")
            model_start_time = time.time()
            model.fit(X_train, y_train)
            model_time = (time.time() - model_start_time) / 60
            print(f"    ✅ {name} completed in {model_time:.2f} minutes")
            results[name] = {
                'model': model,
                'train_metrics': self._calculate_metrics(y_train, model.predict(X_train)),
                'test_metrics': self._calculate_metrics(y_test, model.predict(X_test)),
                'train_predictions': model.predict(X_train),
                'test_predictions': model.predict(X_test),
            }
        return results
    
    def _calculate_metrics(self, y_true, y_pred):
        return {
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }

    def _get_stats_text(self, data, title=""):
        if len(data) == 0: return f"{title}\nN = 0"
        stats = pd.Series(data).describe()
        return (f"{title}\nN = {int(stats['count'])}\nMean = {stats['mean']:.2f}\n"
                f"Std = {stats['std']:.2f}\nMin = {stats['min']:.2f}\nMax = {stats['max']:.2f}")

    def _generate_data_split_distribution_plot(self, y_train, y_test):
        print("  Generating data split distribution plot...")
        plt.figure(figsize=(10, 6))
        sns.histplot(y_train, color="dodgerblue", label='Training Set', kde=True, stat="density", linewidth=0)
        sns.histplot(y_test, color="darkorange", label='Testing Set', kde=True, stat="density", linewidth=0)
        plt.title('Distribution of Depth to Water in Training and Testing Sets')
        plt.xlabel('Depth to Water (m)')
        plt.ylabel('Density')
        props = dict(boxstyle='round', facecolor='white', alpha=0.8)
        plt.text(0.95, 0.95, self._get_stats_text(y_train, 'Training Set'), transform=plt.gca().transAxes, fontsize=10, va='top', ha='right', bbox=props)
        plt.text(0.95, 0.70, self._get_stats_text(y_test, 'Testing Set'), transform=plt.gca().transAxes, fontsize=10, va='top', ha='right', bbox=props)
        plt.legend()
        plt.tight_layout()
        plt.savefig(self.output_dir / 'data_split_distribution.png', facecolor='white')
        plt.close()

    def _generate_temporal_distribution_plots(self):
        print("  Generating temporal distribution plots...")
        if 'Date_String' not in self.df_filtered.columns: 
            print("  Skipping temporal plots: 'Date_String' column not found.")
            return

        df_temp = self.df_filtered[['Date_String', self.target_col]].copy()
        df_temp['Date'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        df_temp.dropna(subset=['Date'], inplace=True)
        df_temp['Year'] = df_temp['Date'].dt.year
        
        train_df = df_temp.loc[self.train_indices]
        test_df = df_temp.loc[self.test_indices]
        fig, axes = plt.subplots(2, 1, figsize=(15, 12))
        counts_df = pd.DataFrame({
            'Training': train_df['Year'].value_counts(), 
            'Testing': test_df['Year'].value_counts()
        }).sort_index().fillna(0)
        counts_df.plot(kind='bar', ax=axes[0], stacked=True, color=['skyblue', 'salmon'])
        axes[0].set_title('Number of Samples per Year')
        axes[0].set_ylabel('Number of Samples')
        axes[0].tick_params(axis='x', rotation=45)
        
        combined_df = pd.concat([train_df.assign(Set='Training'), test_df.assign(Set='Testing')])
        sns.boxplot(x='Year', y=self.target_col, hue='Set', data=combined_df, ax=axes[1], palette={'Training': 'skyblue', 'Testing': 'salmon'})
        axes[1].set_title('Distribution of Depth to Water by Year')
        axes[1].set_ylabel('Depth to Water (m)')
        axes[1].tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.savefig(self.output_dir / 'temporal_distribution.png', facecolor='white')
        plt.close()

    def _generate_one_to_one_plots(self, results, y_train, y_test):
        print("  Generating one-to-one plots...")
        for model_name, res in results.items():
            self._plot_one_to_one(y_train, res['train_predictions'], res['train_metrics'], model_name, 'Training')
            self._plot_one_to_one(y_test, res['test_predictions'], res['test_metrics'], model_name, 'Testing')

    def _plot_one_to_one(self, y_true, y_pred, metrics, model_name, data_type):
        plt.figure(figsize=(8, 8))
        ax = sns.scatterplot(x=y_true, y=y_pred, alpha=0.6, color='steelblue', edgecolor='w', s=50)
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'r--', lw=2, label='1:1 Line')
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel('Actual Depth to Water (m)')
        ax.set_ylabel('Predicted Depth to Water (m)')
        ax.set_title(f'{model_name}: {data_type} Set\nPredicted vs Actual')
        text_str = (f"R² = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.2f} m\n"
                    f"MAPE = {metrics['mape']:.2f}%\nN = {len(y_true)}")
        props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='black')
        ax.text(0.05, 0.95, text_str, transform=ax.transAxes, fontsize=11, va='top', bbox=props)
        plt.legend()
        plt.tight_layout()
        filename = f'{model_name.lower().replace(" ", "_")}_{data_type.lower()}_one_to_one.png'
        plt.savefig(self.output_dir / filename, bbox_inches='tight', facecolor='white')
        plt.close()
        
    def _generate_testing_performance_plots(self, results, y_test):
        print(f"  Generating testing sample performance plots...")
        fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True, sharey=True)
        fig.suptitle(f'Testing Performance (Poor if |Error| > {self.absolute_error_threshold}m)', fontsize=16, fontweight='bold')
        
        for i, (model_name, res) in enumerate(results.items()):
            good_mask, poor_mask = self._classify_performance(y_test, res['test_predictions'])
            ax = axes[i]
            ax.scatter(y_test[good_mask], res['test_predictions'][good_mask], alpha=0.6, color='green', s=30, label=f'Good (N={good_mask.sum()})')
            ax.scatter(y_test[poor_mask], res['test_predictions'][poor_mask], alpha=0.8, color='red', s=40, label=f'Poor (N={poor_mask.sum()})')
            lims = [min(y_test.min(), res['test_predictions'].min()), max(y_test.max(), res['test_predictions'].max())]
            ax.plot(lims, lims, 'k--', lw=2, alpha=0.7, label='1:1 Line')
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            ax.set_xlabel('Actual Depth to Water (m)')
            ax.set_title(f'{model_name}')
            ax.legend()
            if i == 0: ax.set_ylabel('Predicted Depth to Water (m)')
            ax.text(0.02, 0.98, f'{good_mask.sum() / len(y_test) * 100:.1f}% Good', transform=ax.transAxes, fontsize=10, va='top', bbox=dict(boxstyle='round,pad=0.3', fc='lightgreen', alpha=0.7))
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.output_dir / 'testing_performance_scatter.png', facecolor='white')
        plt.close()

    def _generate_performance_distribution_plots(self, results, y_test):
        print("  Generating performance-based distribution plots...")
        fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True, sharey=True)
        fig.suptitle('Distribution of Actual Depths for Good vs. Poor Predictions', fontsize=16, fontweight='bold')
        for i, (model_name, res) in enumerate(results.items()):
            good_mask, poor_mask = self._classify_performance(y_test, res['test_predictions'])
            ax = axes[i]
            sns.histplot(y_test[good_mask], ax=ax, color='green', label='Good', kde=True, stat="density", lw=0, alpha=0.6)
            sns.histplot(y_test[poor_mask], ax=ax, color='red', label='Poor', kde=True, stat="density", lw=0, alpha=0.6)
            ax.set_title(model_name)
            ax.set_xlabel('Actual Depth to Water (m)')
            ax.legend()
            if i == 0: ax.set_ylabel('Density')
            props = dict(boxstyle='round', facecolor='white', alpha=0.8)
            ax.text(0.98, 0.98, self._get_stats_text(y_test[good_mask], 'Good'), transform=ax.transAxes, fontsize=9, va='top', ha='right', bbox=props)
            ax.text(0.98, 0.65, self._get_stats_text(y_test[poor_mask], 'Poor'), transform=ax.transAxes, fontsize=9, va='top', ha='right', bbox=props)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.output_dir / 'performance_distribution_by_model.png', facecolor='white')
        plt.close()

    def _generate_well_location_maps_and_save_results(self, results, y_test):
        print("  Generating well location maps and saving results to Excel...")
        
        # Define possible coordinate column names
        coord_options = [
            ['Longitude_GCS_WGS_1984', 'Latitude_GCS_WGS_1984'],
            ['longitude', 'latitude'],
            ['Longitude', 'Latitude'],
            ['lon', 'lat'],
            ['x', 'y'],
            ['X', 'Y']
        ]
        
        coord_cols = None
        for option in coord_options:
            if all(col in self.df_filtered.columns for col in option):
                coord_cols = option
                break
        
        if coord_cols is None:
            print("  Warning: No coordinate columns found. Skipping map generation.")
            print(f"  Available columns: {list(self.df_filtered.columns)}")
            return
        
        print(f"  Using coordinate columns: {coord_cols}")
        
        # Get test data with coordinates
        test_data_with_coords = self.df_filtered.loc[self.test_indices].copy()
        
        # Check for missing coordinates
        missing_coords = test_data_with_coords[coord_cols].isnull().any(axis=1)
        if missing_coords.any():
            print(f"  Warning: {missing_coords.sum()} test samples have missing coordinates")
            test_data_with_coords = test_data_with_coords[~missing_coords]
        
        if len(test_data_with_coords) == 0:
            print("  Warning: No test samples with valid coordinates. Skipping map generation.")
            return
        
        # Verify coordinate values are reasonable
        lon_col, lat_col = coord_cols
        lon_range = test_data_with_coords[lon_col].min(), test_data_with_coords[lon_col].max()
        lat_range = test_data_with_coords[lat_col].min(), test_data_with_coords[lat_col].max()
        
        print(f"  Coordinate ranges: Lon {lon_range}, Lat {lat_range}")
        
        # Filter results to match available coordinates
        filtered_results = {}
        for model_name, res in results.items():
            # Get predictions only for samples with coordinates
            test_indices_with_coords = test_data_with_coords.index
            mask = np.array([idx in test_indices_with_coords for idx in self.test_indices])
            
            if mask.sum() == 0:
                print(f"  Warning: No matching samples for {model_name}")
                continue
                
            filtered_results[model_name] = {
                'test_predictions': res['test_predictions'][mask]
            }
        
        if not filtered_results:
            print("  Warning: No valid results for mapping. Skipping map generation.")
            return
        
        # Create maps
        excel_data = []
        fig, axes = plt.subplots(1, len(filtered_results), figsize=(8*len(filtered_results), 8))
        if len(filtered_results) == 1:
            axes = [axes]
        
        fig.suptitle(f'Well Performance Map (Poor if |Error| > {self.absolute_error_threshold}m)', fontsize=16, fontweight='bold')
        
        for i, (model_name, res) in enumerate(filtered_results.items()):
            test_data_copy = test_data_with_coords.copy()
            test_data_copy['predicted'] = res['test_predictions']
            
            # Calculate performance
            _, poor_mask = self._classify_performance(test_data_copy[self.target_col], test_data_copy['predicted'])
            test_data_copy['Performance_Category'] = np.where(poor_mask, 'Poor', 'Good')
            
            # Add to Excel data
            excel_data.extend(test_data_copy.assign(Model=model_name).to_dict('records'))

            # Aggregate by well (if 'Code' column exists)
            if 'Code' in test_data_copy.columns:
                well_performance = test_data_copy.groupby('Code').agg({
                    lon_col: 'mean', 
                    lat_col: 'mean',
                    'Performance_Category': lambda x: 'Poor' if 'Poor' in x.values else 'Good'
                }).reset_index()
            else:
                well_performance = test_data_copy[[lon_col, lat_col, 'Performance_Category']].copy()

            ax = axes[i]
            
            # Separate good and poor wells
            good_wells = well_performance[well_performance['Performance_Category'] == 'Good']
            poor_wells = well_performance[well_performance['Performance_Category'] == 'Poor']
            
            print(f"  {model_name}: {len(good_wells)} good wells, {len(poor_wells)} poor wells")

            # Plot with or without basemap
            if BASEMAP_AVAILABLE and len(well_performance) > 0:
                try:
                    # Create GeoDataFrame
                    gdf = gpd.GeoDataFrame(
                        well_performance, 
                        geometry=gpd.points_from_xy(well_performance[lon_col], well_performance[lat_col]), 
                        crs="EPSG:4326"
                    )
                    
                    # Convert to Web Mercator for basemap
                    gdf_mercator = gdf.to_crs(epsg=3857)
                    
                    # Plot points
                    gdf_mercator[gdf_mercator['Performance_Category'] == 'Good'].plot(
                        ax=ax, color='green', alpha=0.7, markersize=60, 
                        label=f'Good Wells (N={len(good_wells)})'
                    )
                    gdf_mercator[gdf_mercator['Performance_Category'] == 'Poor'].plot(
                        ax=ax, color='red', alpha=0.8, markersize=80, 
                        label=f'Poor Wells (N={len(poor_wells)})'
                    )
                    
                    # Add basemap
                    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.8)
                    
                                        # Set axis labels (will be in Web Mercator coordinates)
                    ax.set_xlabel('Easting (m)')
                    ax.set_ylabel('Northing (m)')
                    
                except Exception as e:
                    print(f"  Warning: Basemap error for {model_name}: {e}")
                    # Fall back to simple scatter plot
                    ax.scatter(good_wells[lon_col], good_wells[lat_col], 
                             c='green', s=60, alpha=0.7, label=f'Good Wells (N={len(good_wells)})')
                    ax.scatter(poor_wells[lon_col], poor_wells[lat_col], 
                             c='red', s=80, alpha=0.8, label=f'Poor Wells (N={len(poor_wells)})')
                    ax.set_xlabel(f'{lon_col}')
                    ax.set_ylabel(f'{lat_col}')
            else:
                # Simple scatter plot without basemap
                ax.scatter(good_wells[lon_col], good_wells[lat_col], 
                         c='green', s=60, alpha=0.7, label=f'Good Wells (N={len(good_wells)})')
                ax.scatter(poor_wells[lon_col], poor_wells[lat_col], 
                         c='red', s=80, alpha=0.8, label=f'Poor Wells (N={len(poor_wells)})')
                ax.set_xlabel(f'{lon_col}')
                ax.set_ylabel(f'{lat_col}')
            
            ax.set_title(f'{model_name}')
            ax.legend()
            
            # Add grid
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(self.output_dir / 'well_location_performance_maps.png', facecolor='white', bbox_inches='tight')
        plt.close()
        
        # Save detailed results to Excel
        if excel_data:
            df = pd.DataFrame(excel_data)
            
            # Calculate additional metrics
            df['ape'] = self.absolute_percentage_error(df[self.target_col], df['predicted'])
            df['abs_error'] = np.abs(df[self.target_col] - df['predicted'])
            df['error'] = df[self.target_col] - df['predicted']
            
            # Reorder columns for better readability
            base_cols = ['Model', 'Code'] if 'Code' in df.columns else ['Model']
            coord_cols_final = [col for col in coord_cols if col in df.columns]
            metric_cols = [self.target_col, 'predicted', 'error', 'abs_error', 'ape', 'Performance_Category']
            
            # Get remaining columns
            all_specified = base_cols + coord_cols_final + metric_cols
            other_cols = [col for col in df.columns if col not in all_specified]
            
            # Final column order
            final_col_order = base_cols + coord_cols_final + metric_cols + other_cols
            df = df[final_col_order]
            
            # Save to Excel
            excel_path = self.output_dir / "detailed_testing_results.xlsx"
            try:
                with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                    # Save all results
                    df.to_excel(writer, sheet_name='All_Results', index=False)
                    
                    # Create summary sheet
                    summary_data = []
                    for model_name in df['Model'].unique():
                        model_df = df[df['Model'] == model_name]
                        good_count = (model_df['Performance_Category'] == 'Good').sum()
                        poor_count = (model_df['Performance_Category'] == 'Poor').sum()
                        
                        summary_data.append({
                            'Model': model_name,
                            'Total_Predictions': len(model_df),
                            'Good_Predictions': good_count,
                            'Poor_Predictions': poor_count,
                            'Good_Percentage': good_count / len(model_df) * 100,
                            'Mean_Absolute_Error': model_df['abs_error'].mean(),
                            'Median_Absolute_Error': model_df['abs_error'].median(),
                            'Mean_APE': model_df['ape'][model_df['ape'] != np.inf].mean(),
                            'Median_APE': model_df['ape'][model_df['ape'] != np.inf].median()
                        })
                    
                    summary_df = pd.DataFrame(summary_data)
                    summary_df.to_excel(writer, sheet_name='Summary', index=False)
                    
                    # Format the Excel file
                    workbook = writer.book
                    
                    # Format Summary sheet
                    summary_sheet = writer.sheets['Summary']
                    for column in summary_df:
                        column_width = max(summary_df[column].astype(str).map(len).max(), len(column)) + 2
                        col_idx = summary_df.columns.get_loc(column)
                        summary_sheet.set_column(col_idx, col_idx, column_width)
                    
                    # Format All_Results sheet (limit column width)
                    all_results_sheet = writer.sheets['All_Results']
                    for idx, column in enumerate(df.columns):
                        all_results_sheet.set_column(idx, idx, min(20, max(10, len(column) + 2)))
                
                print(f"  ✅ Detailed results saved to {excel_path}")
                
            except Exception as e:
                print(f"  ❌ Could not save detailed Excel file: {e}")
                # Try to save as CSV as fallback
                try:
                    csv_path = self.output_dir / "detailed_testing_results.csv"
                    df.to_csv(csv_path, index=False)
                    print(f"  ✅ Results saved as CSV to {csv_path}")
                except Exception as e2:
                    print(f"  ❌ Could not save CSV file either: {e2}")

    def generate_feature_importance_plot(self, results, top_n=20):
        """Generate feature importance plots for all models"""
        print("  Generating feature importance plots...")
        
        fig, axes = plt.subplots(1, len(results), figsize=(8*len(results), 10))
        if len(results) == 1:
            axes = [axes]
        
        fig.suptitle('Feature Importance Analysis', fontsize=16, fontweight='bold')
        
        for idx, (model_name, res) in enumerate(results.items()):
            model = res['model']
            
            # Get feature importances
            if hasattr(model, 'feature_importances_'):
                importances = model.feature_importances_
                indices = np.argsort(importances)[::-1][:top_n]
                
                # Get feature names
                feature_names = [self.feature_cols[i] for i in indices]
                importance_values = importances[indices]
                
                # Create horizontal bar plot
                ax = axes[idx]
                y_pos = np.arange(len(feature_names))
                ax.barh(y_pos, importance_values, color='steelblue')
                ax.set_yticks(y_pos)
                ax.set_yticklabels(feature_names)
                ax.set_xlabel('Feature Importance')
                ax.set_title(f'{model_name}')
                ax.grid(True, alpha=0.3, axis='x')
                
                # Add value labels
                for i, v in enumerate(importance_values):
                    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=8)
            else:
                axes[idx].text(0.5, 0.5, f'{model_name}\nNo feature importance available', 
                              ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].set_title(f'{model_name}')
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.output_dir / 'feature_importance.png', facecolor='white', bbox_inches='tight')
        plt.close()

    def generate_prediction_error_analysis(self, results, y_test):
        """Generate detailed error analysis plots"""
        print("  Generating prediction error analysis...")
        
        fig, axes = plt.subplots(2, len(results), figsize=(8*len(results), 12))
        if len(results) == 1:
            axes = axes.reshape(2, 1)
        
        fig.suptitle('Prediction Error Analysis', fontsize=16, fontweight='bold')
        
        for idx, (model_name, res) in enumerate(results.items()):
            predictions = res['test_predictions']
            errors = y_test - predictions
            relative_errors = np.where(y_test != 0, errors / y_test * 100, 0)
            
            # Error vs Actual plot
            ax1 = axes[0, idx]
            scatter = ax1.scatter(y_test, errors, alpha=0.5, c=np.abs(errors), 
                                cmap='RdYlBu_r', s=30)
            ax1.axhline(y=0, color='red', linestyle='--', alpha=0.7)
            ax1.axhline(y=self.absolute_error_threshold, color='orange', linestyle=':', alpha=0.7, label=f'±{self.absolute_error_threshold}m threshold')
            ax1.axhline(y=-self.absolute_error_threshold, color='orange', linestyle=':', alpha=0.7)
            ax1.set_xlabel('Actual Depth to Water (m)')
            ax1.set_ylabel('Prediction Error (m)')
            ax1.set_title(f'{model_name} - Absolute Errors')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            
            # Add colorbar
            cbar = plt.colorbar(scatter, ax=ax1)
            cbar.set_label('|Error| (m)')
            
            # Relative error distribution
            ax2 = axes[1, idx]
            ax2.hist(relative_errors[np.abs(relative_errors) < 200], bins=50, 
                    color='steelblue', alpha=0.7, edgecolor='black')
            ax2.set_xlabel('Relative Error (%)')
            ax2.set_ylabel('Frequency')
            ax2.set_title(f'{model_name} - Relative Error Distribution')
            ax2.axvline(x=0, color='red', linestyle='--', alpha=0.7)
            ax2.grid(True, alpha=0.3)
            
            # Add statistics
            stats_text = f'Mean: {np.mean(errors):.2f}m\nStd: {np.std(errors):.2f}m\nMAE: {np.mean(np.abs(errors)):.2f}m'
            ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, fontsize=9, 
                    va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.output_dir / 'prediction_error_analysis.png', facecolor='white', bbox_inches='tight')
        plt.close()

    def generate_model_comparison_summary(self, results, y_test):
        """Generate a comprehensive model comparison summary plot"""
        print("  Generating model comparison summary...")
        
        # Prepare data for comparison
        model_names = list(results.keys())
        metrics_data = {
            'R²': [],
            'RMSE (m)': [],
            'MAE (m)': [],
            'MAPE (%)': [],
            'Good Predictions (%)': []
        }
        
        for model_name, res in results.items():
            test_metrics = res['test_metrics']
            metrics_data['R²'].append(test_metrics['r2'])
            metrics_data['RMSE (m)'].append(test_metrics['rmse'])
            metrics_data['MAE (m)'].append(test_metrics['mae'])
            metrics_data['MAPE (%)'].append(test_metrics['mape'])
            
            # Calculate good prediction percentage
            good_mask, _ = self._classify_performance(y_test, res['test_predictions'])
            good_pct = good_mask.sum() / len(y_test) * 100
            metrics_data['Good Predictions (%)'].append(good_pct)
        
        # Create subplots
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()
        
        fig.suptitle('Model Performance Comparison - Testing Set', fontsize=16, fontweight='bold')
        
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
        
        for idx, (metric_name, values) in enumerate(metrics_data.items()):
            if idx < 5:  # We have 5 metrics
                ax = axes[idx]
                bars = ax.bar(model_names, values, color=colors[:len(model_names)])
                ax.set_title(metric_name)
                ax.set_ylabel(metric_name)
                ax.grid(True, alpha=0.3, axis='y')
                
                # Add value labels on bars
                for bar, value in zip(bars, values):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height,
                           f'{value:.2f}', ha='center', va='bottom', fontsize=10)
                
                # Rotate x-labels if needed
                if len(model_names) > 3:
                    ax.tick_params(axis='x', rotation=45)
        
        # Remove empty subplot
        fig.delaxes(axes[5])
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(self.output_dir / 'model_comparison_summary.png', facecolor='white', bbox_inches='tight')
        plt.close()

# ==============================================================================
# MAIN EXECUTION BLOCK
# ==============================================================================
def main():
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    if not os.path.exists(file_path):
        print(f"❌ FATAL ERROR: Input file not found at '{file_path}'")
        print("Please update the 'file_path' variable in the main() function.")
        return

    # Initialize and run the predictor
    predictor = GroundwaterPredictor(file_path)
    
    if predictor.load_and_preprocess_data():
        # Run main analysis
        predictor.run_analysis()
        
        # Generate additional analysis plots if results exist
        if hasattr(predictor, 'results') and predictor.results:
            # Get the results from the temporal analysis
            # Note: You might need to store results as class attribute in run_analysis
            print("\n📊 Generating additional analysis plots...")
            
            # These methods would need access to the results from run_analysis
            # You might want to modify run_analysis to store results as self.results

if __name__ == "__main__":
    main()

# Improve the figures

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Set publication-quality parameters
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.major.width'] = 0.8
plt.rcParams['ytick.major.width'] = 0.8
plt.rcParams['xtick.minor.width'] = 0.4
plt.rcParams['ytick.minor.width'] = 0.4
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.edgecolor'] = 'black'
plt.rcParams['legend.fancybox'] = False

def create_improved_performance_visualization():
    """
    Create a 2x3 subplot layout with scatter plots above and distribution plots below
    """
    
    # Path to the CSV file - UPDATE THIS PATH
    csv_path = Path(r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\groundwater_analysis_results_20250711_114958\detailed_testing_results.csv")
    
    # Check if file exists
    if not csv_path.exists():
        # Try alternative name
        csv_path = csv_path.parent / "detailed_testing_results.csv"
        if not csv_path.exists():
            print(f"❌ ERROR: Could not find CSV file at {csv_path}")
            print("Please update the csv_path variable with the correct path.")
            return
    
    print(f"📁 Loading data from: {csv_path}")
    
    # Load the data
    df = pd.read_csv(csv_path)
    
    # Get unique models
    models = df['Model'].unique()
    n_models = len(models)
    
    print(f"📊 Found {n_models} models: {', '.join(models)}")
    
    # Define professional color scheme (colorblind-friendly)
    colors = {
        'Good': '#2E7D32',  # Dark green
        'Poor': '#D32F2F'   # Dark red
    }
    
    # Create figure with 2x3 subplot layout
    fig, axes = plt.subplots(2, n_models, figsize=(18, 10), 
                            gridspec_kw={'height_ratios': [3, 2], 'hspace': 0.35, 'wspace': 0.3})
    
    # Ensure axes is 2D array even for single model
    if n_models == 1:
        axes = axes.reshape(2, 1)
    
    # Main title with better positioning
    fig.suptitle('Model Performance Analysis: Predictions and Actual Depth Distribution',
                 fontsize=15, fontweight='bold', y=0.94)
    
    # Subtitle with criterion
    fig.text(0.5, 0.90, 'Performance criterion: |Error| > 5 m classified as poor prediction',
             ha='center', fontsize=11, style='italic')
    
    # Process each model
    for i, model_name in enumerate(models):
        # Filter data for this model
        model_data = df[df['Model'] == model_name].copy()
        
        # Get actual and predicted values
        actual = model_data['Depth to water (m)'].values
        predicted = model_data['predicted'].values
        performance = model_data['Performance_Category'].values
        
        # Create masks for performance categories
        good_mask = performance == 'Good'
        poor_mask = performance == 'Poor'
        
        # Calculate statistics
        n_good = good_mask.sum()
        n_poor = poor_mask.sum()
        n_total = len(model_data)
        good_pct = (n_good / n_total) * 100
        poor_pct = (n_poor / n_total) * 100
        
        # Calculate R² for all data
        r2_all = np.corrcoef(actual, predicted)[0, 1] ** 2
        
        # TOP ROW: Scatter plots
        ax_scatter = axes[0, i]
        
        # Plot data points
        # Good predictions
        ax_scatter.scatter(actual[good_mask], predicted[good_mask], 
                          c=colors['Good'], s=40, alpha=0.6, edgecolors='none',
                          zorder=5)
        
        # Poor predictions
        ax_scatter.scatter(actual[poor_mask], predicted[poor_mask], 
                          c=colors['Poor'], s=50, alpha=0.8, edgecolors='darkred',
                          linewidth=0.5, zorder=6)
        
        # Add 1:1 line
        lims = [0, max(actual.max(), predicted.max()) * 1.05]
        ax_scatter.plot(lims, lims, 'k-', lw=1.5, alpha=0.8, label='1:1 line', zorder=4)
        
        # Add ±5m error bounds (shaded region)
        ax_scatter.fill_between(lims, [x-5 for x in lims], [x+5 for x in lims], 
                               alpha=0.1, color='gray', label='±5 m bounds', zorder=1)
        
        # Add grid
        ax_scatter.grid(True, which='major', linestyle='-', alpha=0.2, zorder=0)
        ax_scatter.grid(True, which='minor', linestyle=':', alpha=0.1, zorder=0)
        ax_scatter.minorticks_on()
        
        # Set axis properties
        ax_scatter.set_xlim(lims)
        ax_scatter.set_ylim(lims)
        ax_scatter.set_aspect('equal', adjustable='box')
        
        # Labels
        if i == 0:
            ax_scatter.set_ylabel('Predicted depth to water (m)', fontsize=12)
        
        # Model name as title
        model_display_name = model_name.replace('_', ' ')
        ax_scatter.set_title(f'({chr(97+i)}) {model_display_name}', fontsize=13, 
                            fontweight='bold', pad=10)
        
        # Add statistics box with percentages - position it better to avoid legend overlap
        stats_text = f'R² = {r2_all:.3f}\nRMSE = {np.sqrt((model_data["abs_error"]**2).mean()):.2f} m\nMAE = {model_data["abs_error"].mean():.2f} m\nGood: {good_pct:.1f}% (n={n_good})\nPoor: {poor_pct:.1f}% (n={n_poor})'
        
        # Position stats box in lower left to avoid legend overlap
        props = dict(boxstyle='round,pad=0.4', facecolor='white', 
                    edgecolor='black', linewidth=0.8, alpha=0.95)
        ax_scatter.text(0.05, 0.35, stats_text, transform=ax_scatter.transAxes, fontsize=9,
                       verticalalignment='top', bbox=props, fontweight='normal')
        
        # Remove x-axis labels from scatter plots (top row)
        ax_scatter.set_xticklabels([])
        
        # BOTTOM ROW: Distribution plots
        ax_dist = axes[1, i]
        
        # Get actual depths for good and poor predictions
        actual_good = actual[good_mask]
        actual_poor = actual[poor_mask]
        
        # Create histograms with better binning and styling
        bins = np.linspace(0, actual.max() * 1.05, 25)  # Reduced bins for cleaner look
        
        # Plot distributions with better styling
        n_good_hist, bins_good, patches_good = ax_dist.hist(actual_good, bins=bins, 
                                                           color=colors['Good'], alpha=0.8, 
                                                           density=True, edgecolor='darkgreen', 
                                                           linewidth=0.5, 
                                                           label=f'Good (n={n_good}, {good_pct:.1f}%)', 
                                                           zorder=3)
        n_poor_hist, bins_poor, patches_poor = ax_dist.hist(actual_poor, bins=bins, 
                                                           color=colors['Poor'], alpha=0.8, 
                                                           density=True, edgecolor='darkred', 
                                                           linewidth=0.5,
                                                           label=f'Poor (n={n_poor}, {poor_pct:.1f}%)', 
                                                           zorder=3)
        
        # Add grid
        ax_dist.grid(True, which='major', linestyle='-', alpha=0.2, axis='y', zorder=0)
        ax_dist.grid(True, which='minor', linestyle=':', alpha=0.1, axis='y', zorder=0)
        ax_dist.minorticks_on()
        
        # Set labels
        ax_dist.set_xlabel('Actual Depth to Water (m)', fontsize=12)
        if i == 0:
            ax_dist.set_ylabel('Density', fontsize=12)
        
        # Title for distribution plots
        ax_dist.set_title(f'({chr(100+i)}) Distribution of Actual Depths', 
                         fontsize=13, fontweight='bold', pad=10)
        
        # Add statistics text box for distributions - make it more compact and better positioned
        good_stats = f'Good: μ={actual_good.mean():.1f}±{actual_good.std():.1f}m'
        poor_stats = f'Poor: μ={actual_poor.mean():.1f}±{actual_poor.std():.1f}m'
        total_stats = f'Total: N={n_total}'
        
        dist_stats_text = f'{good_stats}\n{poor_stats}\n{total_stats}'
        
        props_dist = dict(boxstyle='round,pad=0.4', facecolor='white', 
                         edgecolor='black', linewidth=0.8, alpha=0.95)
        ax_dist.text(0.98, 0.98, dist_stats_text, transform=ax_dist.transAxes, 
                    fontsize=9, verticalalignment='top', horizontalalignment='right', 
                    bbox=props_dist, fontweight='normal')
        
        # Add ticks
        ax_dist.tick_params(which='both', direction='in', top=True, right=True)
        ax_dist.tick_params(which='major', length=5, width=0.8)
        ax_dist.tick_params(which='minor', length=3, width=0.4)
        
        # Add legend to first distribution subplot with better positioning
        if i == 0:
            legend = ax_dist.legend(loc='upper right', frameon=True, fancybox=False,
                                  edgecolor='black', framealpha=0.95, borderpad=0.4,
                                  fontsize=9)
            # Adjust legend position to avoid overlap with stats
            legend.set_bbox_to_anchor((0.6, 0.98))
    
    # Create shared legend for scatter plots (place it in upper right of figure)
    # Create legend elements for scatter plots
    good_patch = mpatches.Circle((0, 0), 1, facecolor=colors['Good'], 
                               edgecolor='none', alpha=0.6, 
                               label='Good prediction')
    poor_patch = mpatches.Circle((0, 0), 1, facecolor=colors['Poor'], 
                               edgecolor='darkred', linewidth=0.5,
                               alpha=0.8, label='Poor prediction')
    line_patch = mpatches.Patch(color='black', label='1:1 line')
    bounds_patch = mpatches.Patch(color='gray', alpha=0.3, 
                                label='±5 m bounds')
    
    # Place legend in upper right corner, with better positioning
    fig.legend(handles=[good_patch, poor_patch, line_patch, bounds_patch],
              loc='upper right', bbox_to_anchor=(0.99, 0.86),
              frameon=True, fancybox=False, edgecolor='black', 
              framealpha=0.95, borderpad=0.4, fontsize=9, 
              title='Scatter Plot Legend', title_fontsize=9)
    
    # Adjust layout to make room for legends and prevent overlap
    plt.subplots_adjust(left=0.06, right=0.94, top=0.85, bottom=0.08)
    
    # Save the figure
    output_path = Path("improved_model_performance_analysis.png")
    plt.savefig(output_path, dpi=600, bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    print(f"✅ Figure saved as: {output_path}")
    
    # Also save as PDF for vector graphics
    pdf_path = output_path.with_suffix('.pdf')
    plt.savefig(pdf_path, format='pdf', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    print(f"✅ PDF version saved as: {pdf_path}")
    
    # Also save as EPS
    eps_path = output_path.with_suffix('.eps')
    plt.savefig(eps_path, format='eps', bbox_inches='tight',
                facecolor='white', edgecolor='none')
    print(f"✅ EPS version saved as: {eps_path}")
    
    plt.close()
    
    # Generate enhanced summary table
    print("\n📊 Enhanced Summary Statistics for Paper:")
    print("="*80)
    summary_data = []
    
    for model_name in models:
        model_data = df[df['Model'] == model_name]
        good_mask = model_data['Performance_Category'] == 'Good'
        poor_mask = model_data['Performance_Category'] == 'Poor'
        
        actual = model_data['Depth to water (m)'].values
        actual_good = actual[good_mask]
        actual_poor = actual[poor_mask]
        
        summary_data.append({
            'Model': model_name,
            'Total_N': len(model_data),
            'Good_N': good_mask.sum(),
            'Good_%': (good_mask.sum() / len(model_data)) * 100,
            'Poor_N': poor_mask.sum(),
            'Poor_%': (poor_mask.sum() / len(model_data)) * 100,
            'RMSE_m': np.sqrt((model_data['abs_error']**2).mean()),
            'MAE_m': model_data['abs_error'].mean(),
            'R²': np.corrcoef(actual, model_data['predicted'])[0, 1]**2,
            'Good_Depth_Mean': actual_good.mean() if len(actual_good) > 0 else np.nan,
            'Good_Depth_Std': actual_good.std() if len(actual_good) > 0 else np.nan,
            'Poor_Depth_Mean': actual_poor.mean() if len(actual_poor) > 0 else np.nan,
            'Poor_Depth_Std': actual_poor.std() if len(actual_poor) > 0 else np.nan
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # Display formatted summary
    display_cols = ['Model', 'Total_N', 'Good_N', 'Good_%', 'Poor_N', 'Poor_%', 
                   'RMSE_m', 'MAE_m', 'R²']
    print(summary_df[display_cols].to_string(index=False, float_format='%.2f'))
    
    print("\nDepth Distribution Statistics:")
    print("-" * 50)
    depth_cols = ['Model', 'Good_Depth_Mean', 'Good_Depth_Std', 
                 'Poor_Depth_Mean', 'Poor_Depth_Std']
    print(summary_df[depth_cols].to_string(index=False, float_format='%.2f'))
    
    # Save enhanced summary to CSV
    summary_df.to_csv('enhanced_model_performance_summary.csv', index=False, float_format='%.3f')
    print(f"\n✅ Enhanced summary statistics saved to: enhanced_model_performance_summary.csv")

if __name__ == "__main__":
    print("🎨 Generating improved model performance visualization...")
    print("="*60)
    
    # Generate improved figure
    create_improved_performance_visualization()
    
    print("\n✅ Improved visualization generated successfully!")
    print("\nNote: The figure now includes:")
    print("- Top row: Scatter plots with prediction vs actual depths")
    print("- Bottom row: Distribution of actual depths for good vs poor predictions")
    print("- Enhanced statistics including percentages and sample sizes")
    print("- Better layout with shared legends")
    print("\nFor Journal submission, use the PDF or EPS versions for best quality.")

## Google

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Set publication-quality parameters
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.major.width'] = 0.8
plt.rcParams['ytick.major.width'] = 0.8
plt.rcParams['xtick.minor.width'] = 0.4
plt.rcParams['ytick.minor.width'] = 0.4
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.edgecolor'] = 'black'
plt.rcParams['legend.fancybox'] = False

def create_improved_performance_visualization():
    """
    Create a 2x3 subplot layout with scatter plots above and distribution plots below
    """
    
    # Path to the CSV file - UPDATE THIS PATH
    csv_path = Path(r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\groundwater_analysis_results_20250711_114958\detailed_testing_results.csv")
    
    # Check if file exists
    if not csv_path.exists():
        print(f"❌ ERROR: Could not find CSV file at {csv_path}")
        print("Please update the csv_path variable with the correct path.")
        # As a fallback for demonstration, create a dummy dataframe
        print("Creating a dummy dataframe for demonstration purposes.")
        data = {
            'Model': np.repeat(['Extra Trees', 'Random Forest', 'Decision Tree'], 300),
            'Depth to water (m)': np.random.rand(900) * 150,
            'predicted': np.random.rand(900) * 150,
        }
        df = pd.DataFrame(data)
        df['abs_error'] = np.abs(df['predicted'] - df['Depth to water (m)'])
        df['Performance_Category'] = np.where(df['abs_error'] > 5, 'Poor', 'Good')
    else:
        print(f"📁 Loading data from: {csv_path}")
        # Load the data
        df = pd.read_csv(csv_path)

    # Get unique models
    models = df['Model'].unique()
    n_models = len(models)
    
    print(f"📊 Found {n_models} models: {', '.join(models)}")
    
    # Define professional color scheme (colorblind-friendly)
    colors = {
        'Good': '#2E7D32',  # Dark green
        'Poor': '#D32F2F'   # Dark red
    }
    
    # Create figure with 2x3 subplot layout
    fig, axes = plt.subplots(2, n_models, figsize=(18, 10), 
                            gridspec_kw={'height_ratios': [3, 2], 'hspace': 0.35, 'wspace': 0.3})
    
    # Ensure axes is 2D array even for single model
    if n_models == 1:
        axes = axes.reshape(2, 1)
    
    # Main title with better positioning
    fig.suptitle('Model Performance Analysis: Predictions and Actual Depth Distribution',
                 fontsize=15, fontweight='bold', y=0.94)
    
    # Subtitle with criterion
    fig.text(0.5, 0.90, 'Performance criterion: |Error| > 5 m classified as poor prediction',
             ha='center', fontsize=11, style='italic')
    
    # Process each model
    for i, model_name in enumerate(models):
        # Filter data for this model
        model_data = df[df['Model'] == model_name].copy()
        
        # Get actual and predicted values
        actual = model_data['Depth to water (m)'].values
        predicted = model_data['predicted'].values
        performance = model_data['Performance_Category'].values
        
        # Create masks for performance categories
        good_mask = performance == 'Good'
        poor_mask = performance == 'Poor'
        
        # Calculate statistics
        n_good = good_mask.sum()
        n_poor = poor_mask.sum()
        n_total = len(model_data)
        good_pct = (n_good / n_total) * 100
        poor_pct = (n_poor / n_total) * 100
        
        # Calculate R² for all data
        r2_all = np.corrcoef(actual, predicted)[0, 1] ** 2
        
        # TOP ROW: Scatter plots
        ax_scatter = axes[0, i]
        
        # Plot data points
        ax_scatter.scatter(actual[good_mask], predicted[good_mask], 
                          c=colors['Good'], s=40, alpha=0.6, edgecolors='none', zorder=5)
        ax_scatter.scatter(actual[poor_mask], predicted[poor_mask], 
                          c=colors['Poor'], s=50, alpha=0.8, edgecolors='darkred',
                          linewidth=0.5, zorder=6)
        
        # Add 1:1 line
        lims = [0, max(actual.max(), predicted.max()) * 1.05]
        ax_scatter.plot(lims, lims, 'k-', lw=1.5, alpha=0.8, label='1:1 line', zorder=4)
        
        # Add ±5m error bounds (shaded region)
        ax_scatter.fill_between(lims, [x-5 for x in lims], [x+5 for x in lims], 
                               alpha=0.1, color='gray', label='±5 m bounds', zorder=1)
        
        ax_scatter.grid(True, which='major', linestyle='-', alpha=0.2, zorder=0)
        ax_scatter.minorticks_on()
        
        ax_scatter.set_xlim(lims)
        ax_scatter.set_ylim(lims)
        ax_scatter.set_aspect('equal', adjustable='box')
        
        if i == 0:
            ax_scatter.set_ylabel('Predicted depth to water (m)', fontsize=12)
        
        model_display_name = model_name.replace('_', ' ')
        ax_scatter.set_title(f'({chr(97+i)}) {model_display_name}', fontsize=13, 
                            fontweight='bold', pad=10)
        
        stats_text = f'R² = {r2_all:.3f}\nRMSE = {np.sqrt((model_data["abs_error"]**2).mean()):.2f} m\nMAE = {model_data["abs_error"].mean():.2f} m\nGood: {good_pct:.1f}% (n={n_good})\nPoor: {poor_pct:.1f}% (n={n_poor})'
        
        props = dict(boxstyle='round,pad=0.4', facecolor='white', 
                    edgecolor='black', linewidth=0.8, alpha=0.95)
        ax_scatter.text(0.05, 0.95, stats_text, transform=ax_scatter.transAxes, fontsize=9,
                       verticalalignment='top', bbox=props, fontweight='normal')
        
        # BOTTOM ROW: Distribution plots
        ax_dist = axes[1, i]
        
        actual_good = actual[good_mask]
        actual_poor = actual[poor_mask]
        
        bins = np.linspace(0, max(1, actual.max()) * 1.05, 25)
        
        # Plot distributions without percentage in the label
        ax_dist.hist(actual_good, bins=bins, color=colors['Good'], alpha=0.8, 
                     density=True, edgecolor='darkgreen', linewidth=0.5, 
                     label=f'Good (n={n_good})', zorder=3)
        ax_dist.hist(actual_poor, bins=bins, color=colors['Poor'], alpha=0.8, 
                     density=True, edgecolor='darkred', linewidth=0.5,
                     label=f'Poor (n={n_poor})', zorder=3)
        
        ax_dist.grid(True, which='major', linestyle='-', alpha=0.2, axis='y', zorder=0)
        ax_dist.minorticks_on()
        
        ax_dist.set_xlabel('Actual Depth to Water (m)', fontsize=12)
        if i == 0:
            ax_dist.set_ylabel('Density', fontsize=12)
        
        ax_dist.set_title(f'({chr(100+i)}) Distribution of Actual Depths', 
                         fontsize=13, fontweight='bold', pad=10)
        
        good_stats = f'Good: μ={actual_good.mean():.1f}±{actual_good.std():.1f}m'
        poor_stats = f'Poor: μ={actual_poor.mean():.1f}±{actual_poor.std():.1f}m'
        total_stats = f'Total: N={n_total}'
        dist_stats_text = f'{good_stats}\n{poor_stats}\n{total_stats}'
        
        props_dist = dict(boxstyle='round,pad=0.4', facecolor='white', 
                         edgecolor='black', linewidth=0.8, alpha=0.95)
        ax_dist.text(0.98, 0.98, dist_stats_text, transform=ax_dist.transAxes, 
                    fontsize=9, verticalalignment='top', horizontalalignment='right', 
                    bbox=props_dist, fontweight='normal')
        
        if i == 0:
            ax_dist.legend(loc='upper center', frameon=True, fancybox=False,
                           edgecolor='black', framealpha=0.95, borderpad=0.4,
                           fontsize=9)

    # Create shared legend for scatter plots and place it in the top-left corner
    good_patch = mpatches.Patch(color=colors['Good'], alpha=0.7, label='Good prediction')
    poor_patch = mpatches.Patch(color=colors['Poor'], alpha=0.8, label='Poor prediction')
    line_patch = mpatches.Patch(color='black', label='1:1 line')
    bounds_patch = mpatches.Patch(color='gray', alpha=0.3, label='±5 m bounds')
    
    # Place legend in upper left corner of the figure
    fig.legend(handles=[good_patch, poor_patch, line_patch, bounds_patch],
              loc='upper left', bbox_to_anchor=(0.06, 0.86),
              frameon=True, fancybox=False, edgecolor='black', 
              framealpha=0.95, borderpad=0.4, fontsize=9, 
              title='Scatter Plot Legend', title_fontsize=9)
    
    plt.subplots_adjust(left=0.06, right=0.98, top=0.85, bottom=0.08)
    
    # Save the figure
    output_path = Path("improved_model_performance_analysis.png")
    plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✅ Figure saved as: {output_path}")
    
    plt.show()


if __name__ == "__main__":
    print("🎨 Generating improved model performance visualization...")
    create_improved_performance_visualization()
    print("\n✅ Improved visualization generated successfully!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Set publication-quality parameters
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.major.width'] = 0.8
plt.rcParams['ytick.major.width'] = 0.8
plt.rcParams['xtick.minor.width'] = 0.4
plt.rcParams['ytick.minor.width'] = 0.4
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.edgecolor'] = 'black'
plt.rcParams['legend.fancybox'] = False

def create_improved_performance_visualization():
    """
    Create a 2x3 subplot layout with scatter plots above and distribution plots below
    """
    
    # Path to the CSV file - UPDATE THIS PATH
    csv_path = Path(r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\groundwater_analysis_results_20250711_114958\detailed_testing_results.csv")
    
    # Check if file exists
    if not csv_path.exists():
        print(f"❌ ERROR: Could not find CSV file at {csv_path}")
        print("Please update the csv_path variable with the correct path.")
        # As a fallback for demonstration, create a dummy dataframe
        print("Creating a dummy dataframe for demonstration purposes.")
        data = {
            'Model': np.repeat(['Extra Trees', 'Random Forest', 'Decision Tree'], 300),
            'Depth to water (m)': np.random.rand(900) * 150,
            'predicted': np.random.rand(900) * 150,
        }
        df = pd.DataFrame(data)
        df['abs_error'] = np.abs(df['predicted'] - df['Depth to water (m)'])
        df['Performance_Category'] = np.where(df['abs_error'] > 5, 'Poor', 'Good')
    else:
        print(f"📁 Loading data from: {csv_path}")
        # Load the data
        df = pd.read_csv(csv_path)

    # Get unique models
    models = df['Model'].unique()
    n_models = len(models)
    
    print(f"📊 Found {n_models} models: {', '.join(models)}")
    
    # Define professional color scheme (colorblind-friendly)
    colors = {
        'Good': '#2E7D32',  # Dark green
        'Poor': '#D32F2F'   # Dark red
    }
    
    # Create figure with 2x3 subplot layout
    fig, axes = plt.subplots(2, n_models, figsize=(18, 10), 
                            gridspec_kw={'height_ratios': [3, 2], 'hspace': 0.35, 'wspace': 0.3})
    
    # Ensure axes is 2D array even for single model
    if n_models == 1:
        axes = axes.reshape(2, 1)
    
    # Main title with better positioning
    fig.suptitle('Model Performance Analysis: Predictions and Actual Depth Distribution',
                 fontsize=15, fontweight='bold', y=0.96)
    
    # Subtitle with criterion
    fig.text(0.5, 0.92, 'Performance criterion: |Error| > 5 m classified as poor prediction',
             ha='center', fontsize=11, style='italic')
    
    # Process each model
    for i, model_name in enumerate(models):
        # Filter data for this model
        model_data = df[df['Model'] == model_name].copy()
        
        # Get actual and predicted values
        actual = model_data['Depth to water (m)'].values
        predicted = model_data['predicted'].values
        performance = model_data['Performance_Category'].values
        
        # Create masks for performance categories
        good_mask = performance == 'Good'
        poor_mask = performance == 'Poor'
        
        # Calculate statistics
        n_good = good_mask.sum()
        n_poor = poor_mask.sum()
        n_total = len(model_data)
        good_pct = (n_good / n_total) * 100
        poor_pct = (n_poor / n_total) * 100
        
        # Calculate R² for all data
        r2_all = np.corrcoef(actual, predicted)[0, 1] ** 2
        
        # TOP ROW: Scatter plots
        ax_scatter = axes[0, i]
        
        # Plot data points
        ax_scatter.scatter(actual[good_mask], predicted[good_mask], 
                          c=colors['Good'], s=40, alpha=0.6, edgecolors='none', zorder=5)
        ax_scatter.scatter(actual[poor_mask], predicted[poor_mask], 
                          c=colors['Poor'], s=50, alpha=0.8, edgecolors='darkred',
                          linewidth=0.5, zorder=6)
        
        # Add 1:1 line
        lims = [0, max(actual.max(), predicted.max()) * 1.05]
        ax_scatter.plot(lims, lims, 'k-', lw=1.5, alpha=0.8, label='1:1 line', zorder=4)
        
        # Add ±5m error bounds (shaded region)
        ax_scatter.fill_between(lims, [x-5 for x in lims], [x+5 for x in lims], 
                               alpha=0.1, color='gray', label='±5 m bounds', zorder=1)
        
        ax_scatter.grid(True, which='major', linestyle='-', alpha=0.2, zorder=0)
        ax_scatter.minorticks_on()
        
        ax_scatter.set_xlim(lims)
        ax_scatter.set_ylim(lims)
        ax_scatter.set_aspect('equal', adjustable='box')
        
        if i == 0:
            ax_scatter.set_ylabel('Predicted depth to water (m)', fontsize=12)
        
        model_display_name = model_name.replace('_', ' ')
        ax_scatter.set_title(f'({chr(97+i)}) {model_display_name}', fontsize=13, 
                            fontweight='bold', pad=10)
        
        stats_text = f'R² = {r2_all:.3f}\nRMSE = {np.sqrt((model_data["abs_error"]**2).mean()):.2f} m\nMAE = {model_data["abs_error"].mean():.2f} m\nGood: {good_pct:.1f}% (n={n_good})\nPoor: {poor_pct:.1f}% (n={n_poor})'
        
        props = dict(boxstyle='round,pad=0.4', facecolor='white', 
                    edgecolor='black', linewidth=0.8, alpha=0.95)
        ax_scatter.text(0.05, 0.95, stats_text, transform=ax_scatter.transAxes, fontsize=9,
                       verticalalignment='top', bbox=props, fontweight='normal')
        
        # BOTTOM ROW: Distribution plots
        ax_dist = axes[1, i]
        
        actual_good = actual[good_mask]
        actual_poor = actual[poor_mask]
        
        bins = np.linspace(0, max(1, actual.max()) * 1.05, 25)
        
        # Plot distributions
        ax_dist.hist(actual_good, bins=bins, color=colors['Good'], alpha=0.8, 
                     density=True, edgecolor='darkgreen', linewidth=0.5, zorder=3)
        ax_dist.hist(actual_poor, bins=bins, color=colors['Poor'], alpha=0.8, 
                     density=True, edgecolor='darkred', linewidth=0.5, zorder=3)
        
        ax_dist.grid(True, which='major', linestyle='-', alpha=0.2, axis='y', zorder=0)
        ax_dist.minorticks_on()
        
        ax_dist.set_xlabel('Actual Depth to Water (m)', fontsize=12)
        if i == 0:
            ax_dist.set_ylabel('Density', fontsize=12)
        
        ax_dist.set_title(f'({chr(100+i)}) Distribution of Actual Depths', 
                         fontsize=13, fontweight='bold', pad=10)
        
        good_stats = f'Good: μ={actual_good.mean():.1f}±{actual_good.std():.1f}m'
        poor_stats = f'Poor: μ={actual_poor.mean():.1f}±{actual_poor.std():.1f}m'
        total_stats = f'Total: N={n_total}'
        dist_stats_text = f'{good_stats}\n{poor_stats}\n{total_stats}'
        
        props_dist = dict(boxstyle='round,pad=0.4', facecolor='white', 
                         edgecolor='black', linewidth=0.8, alpha=0.95)
        ax_dist.text(0.98, 0.98, dist_stats_text, transform=ax_dist.transAxes, 
                    fontsize=9, verticalalignment='top', horizontalalignment='right', 
                    bbox=props_dist, fontweight='normal')
        
        # The redundant legend from the distribution plot has been removed.

    # Create shared legend for scatter plots
    good_patch = mpatches.Patch(color=colors['Good'], alpha=0.7, label='Good prediction')
    poor_patch = mpatches.Patch(color=colors['Poor'], alpha=0.8, label='Poor prediction')
    line_patch = mpatches.Patch(color='black', label='1:1 line')
    bounds_patch = mpatches.Patch(color='gray', alpha=0.3, label='±5 m bounds')
    
    # Place legend in upper right corner of the figure to avoid overlap
    fig.legend(handles=[good_patch, poor_patch, line_patch, bounds_patch],
              loc='upper right', bbox_to_anchor=(0.95, 0.88),
              frameon=True, fancybox=False, edgecolor='black', 
              framealpha=0.95, borderpad=0.4, fontsize=9, 
              title='Scatter Plot Legend', title_fontsize=9)
    
    plt.subplots_adjust(left=0.06, right=0.98, top=0.88, bottom=0.08)
    
    # Save the figure
    output_path = Path("improved_model_performance_analysis.png")
    plt.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"✅ Figure saved as: {output_path}")
    
    plt.show()


if __name__ == "__main__":
    print("🎨 Generating improved model performance visualization...")
    create_improved_performance_visualization()
    print("\n✅ Improved visualization generated successfully!")

# ArcGIS Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def create_location_performance_csvs(input_csv_path, output_dir="."):
    """
    Create two CSV files from the testing results:
    1. All testing points with location and performance
    2. Unique locations with aggregated performance
    
    Parameters:
    input_csv_path (str): Path to the detailed testing results CSV
    output_dir (str): Directory to save output CSV files
    """
    
    # Check if input file exists
    input_path = Path(input_csv_path)
    if not input_path.exists():
        print(f"❌ ERROR: Could not find CSV file at {input_path}")
        print("Please update the input_csv_path with the correct path.")
        return
    
    print(f"📁 Loading data from: {input_path}")
    
    try:
        # Load the data
        df = pd.read_csv(input_path)
        print(f"✅ Successfully loaded {len(df)} records")
        
        # Display basic info about the dataset
        print(f"📊 Dataset info:")
        print(f"   - Models: {df['Model'].unique()}")
        print(f"   - Performance categories: {df['Performance_Category'].unique()}")
        print(f"   - Date range: {df['Date_String'].min()} to {df['Date_String'].max()}")
        
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return
    
    # Create output directory if it doesn't exist
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    # ================================================================
    # CSV 1: All testing points with location and performance
    # ================================================================
    
    # Select relevant columns for the first CSV
    location_performance_cols = [
        'Model',
        'Code', 
        'Longitude_GCS_WGS_1984', 
        'Latitude_GCS_WGS_1984',
        'Performance_Category',
        'ID',
        'Name',
        'Basin',
        'Sub_Basin',
        'Date_String',
        'Depth to water (m)',
        'predicted',
        'abs_error'
    ]
    
    # Create first CSV with all testing points
    df_all_points = df[location_performance_cols].copy()
    
    # Sort by Model, then by Code, then by Date
    df_all_points = df_all_points.sort_values(['Model', 'Code', 'Date_String'])
    
    output_file_1 = output_path / "testing_points_location_performance.csv"
    df_all_points.to_csv(output_file_1, index=False)
    
    print(f"✅ CSV 1 created: {output_file_1}")
    print(f"   - Contains {len(df_all_points)} testing point records")
    print(f"   - Columns: {list(df_all_points.columns)}")
    
    # ================================================================
    # CSV 2: Unique locations with aggregated performance
    # ================================================================
    
    # Group by location (Code) and Model to aggregate performance
    location_groups = df.groupby(['Model', 'Code', 'Longitude_GCS_WGS_1984', 
                                  'Latitude_GCS_WGS_1984', 'Name', 'Basin', 'Sub_Basin'])
    
    aggregated_data = []
    
    for (model, code, lon, lat, name, basin, sub_basin), group in location_groups:
        # Get unique performance categories for this location
        performance_cats = group['Performance_Category'].unique()
        
        # Determine aggregated performance
        if len(performance_cats) == 1:
            # Only one type of performance
            agg_performance = performance_cats[0]
        else:
            # Mixed performance
            agg_performance = "Both"
        
        # Calculate additional statistics
        n_measurements = len(group)
        good_count = (group['Performance_Category'] == 'Good').sum()
        poor_count = (group['Performance_Category'] == 'Poor').sum()
        good_percentage = (good_count / n_measurements) * 100
        
        # Get date range
        date_range_start = group['Date_String'].min()
        date_range_end = group['Date_String'].max()
        
        # Calculate depth statistics
        mean_depth = group['Depth to water (m)'].mean()
        std_depth = group['Depth to water (m)'].std()
        mean_abs_error = group['abs_error'].mean()
        
        aggregated_data.append({
            'Model': model,
            'Code': code,
            'Longitude_GCS_WGS_1984': lon,
            'Latitude_GCS_WGS_1984': lat,
            'Name': name,
            'Basin': basin,
            'Sub_Basin': sub_basin,
            'Aggregated_Performance': agg_performance,
            'Total_Measurements': n_measurements,
            'Good_Count': good_count,
            'Poor_Count': poor_count,
            'Good_Percentage': round(good_percentage, 1),
            'Date_Range_Start': date_range_start,
            'Date_Range_End': date_range_end,
            'Mean_Depth_m': round(mean_depth, 2),
            'Std_Depth_m': round(std_depth, 2) if not pd.isna(std_depth) else 0.0,
            'Mean_Absolute_Error_m': round(mean_abs_error, 2)
        })
    
    # Create DataFrame for unique locations
    df_unique_locations = pd.DataFrame(aggregated_data)
    
    # Sort by Model, then by Aggregated_Performance, then by Code
    df_unique_locations = df_unique_locations.sort_values([
        'Model', 'Aggregated_Performance', 'Code'
    ])
    
    output_file_2 = output_path / "unique_locations_aggregated_performance.csv"
    df_unique_locations.to_csv(output_file_2, index=False)
    
    print(f"✅ CSV 2 created: {output_file_2}")
    print(f"   - Contains {len(df_unique_locations)} unique location records")
    print(f"   - Columns: {list(df_unique_locations.columns)}")
    
    # ================================================================
    # Summary Statistics
    # ================================================================
    
    print(f"\n📈 Summary Statistics:")
    print("="*50)
    
    # Performance distribution for all points
    perf_dist_all = df['Performance_Category'].value_counts()
    print(f"All testing points performance distribution:")
    for perf, count in perf_dist_all.items():
        pct = (count / len(df)) * 100
        print(f"   {perf}: {count} ({pct:.1f}%)")
    
    # Performance distribution for unique locations
    print(f"\nUnique locations performance distribution:")
    for model in df_unique_locations['Model'].unique():
        model_data = df_unique_locations[df_unique_locations['Model'] == model]
        print(f"  {model}:")
        agg_dist = model_data['Aggregated_Performance'].value_counts()
        for perf, count in agg_dist.items():
            pct = (count / len(model_data)) * 100
            print(f"     {perf}: {count} ({pct:.1f}%)")
    
    # Basin distribution
    print(f"\nBasin distribution:")
    basin_dist = df_unique_locations['Basin'].value_counts().head(10)
    for basin, count in basin_dist.items():
        print(f"   {basin}: {count} locations")
    
    print(f"\n✅ Analysis complete! Files saved to: {output_path}")
    
    return df_all_points, df_unique_locations


# Example usage
if __name__ == "__main__":
    # UPDATE THIS PATH to point to your detailed testing results CSV
    input_csv_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\groundwater_analysis_results_20250711_114958\detailed_testing_results.csv"
    
    # Optional: specify output directory
    output_directory = "."  # Current directory
    
    print("🎯 Creating location performance CSV files...")
    print("="*60)
    
    try:
        df_all, df_unique = create_location_performance_csvs(
            input_csv_path=input_csv_path,
            output_dir=output_directory
        )
        
        print(f"\n🎉 Success! Two CSV files have been created:")
        print(f"   1. testing_points_location_performance.csv")
        print(f"   2. unique_locations_aggregated_performance.csv")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Please check the input file path and try again.")